# LeetCode #363: Max Sum of Rectangle No Larger Than K

https://leetcode.com/problems/max-sum-of-rectangle-no-larger-than-k/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (Enumerate All Rectangles)** | $O(m^2 \cdot n^2 \cdot m)$ | $O(m \cdot n)$ |
| **Optimal: Prefix Sums + Sorted Set ★** | $O(m^2 \cdot n \log n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force (Enumerate All Rectangles)
Fix all four boundaries of the rectangle and compute the sum using a 2D prefix sum. Check every possible sub-rectangle and track the maximum sum that does not exceed k.

### Optimal: Prefix Sums + Sorted Set ★
Fix two column (or row) boundaries and compress the 2D problem into a 1D problem: compute a running prefix sum along the remaining dimension. For each prefix sum `S`, we need the smallest prefix sum `S'` in the sorted set such that `S - S' <= k`, i.e., `S' >= S - k`. Use a sorted container to binary-search for this value in $O(\log n)$ time.

**Why this is better than Brute Force:** By reducing the inner enumeration to a sorted-set lookup, we cut the per-column-pair work from $O(n^2)$ to $O(n \log n)$, giving a significant speedup when the matrix is large.

**Constraints:**
* m == matrix.length, n == matrix[i].length
* 1 <= m, n <= 100
* -100 <= matrix[i][j] <= 100
* -10^5 <= k <= 10^5

## Solutions

### C#

In [ ]:
public class Solution {
    public int MaxSumSubmatrix(int[][] matrix, int k) {
        int m = matrix.Length, n = matrix[0].Length;
        int ans = int.MinValue;
        for (int l = 0; l < n; l++) {
            int[] rowSum = new int[m];
            for (int r = l; r < n; r++) {
                for (int i = 0; i < m; i++)
                    rowSum[i] += matrix[i][r];
                var sorted = new SortedSet<int> { 0 };
                int prefix = 0;
                foreach (int v in rowSum) {
                    prefix += v;
                    var view = sorted.GetViewBetween(prefix - k, int.MaxValue);
                    if (view.Count > 0)
                        ans = Math.Max(ans, prefix - view.Min);
                    sorted.Add(prefix);
                }
            }
        }
        return ans;
    }
}

### Python

In [ ]:
from sortedcontainers import SortedList

class Solution:
    def maxSumSubmatrix(self, matrix: list[list[int]], k: int) -> int:
        m, n = len(matrix), len(matrix[0])
        ans = float('-inf')
        for l in range(n):
            row_sum = [0] * m
            for r in range(l, n):
                for i in range(m):
                    row_sum[i] += matrix[i][r]
                sl = SortedList([0])
                prefix = 0
                for v in row_sum:
                    prefix += v
                    idx = sl.bisect_left(prefix - k)
                    if idx < len(sl):
                        ans = max(ans, prefix - sl[idx])
                    sl.add(prefix)
        return ans

### Go

In [ ]:
import "sort"

func maxSumSubmatrix(matrix [][]int, k int) int {
    m, n := len(matrix), len(matrix[0])
    ans := -1 << 31
    for l := 0; l < n; l++ {
        rowSum := make([]int, m)
        for r := l; r < n; r++ {
            for i := 0; i < m; i++ {
                rowSum[i] += matrix[i][r]
            }
            sorted := []int{0}
            prefix := 0
            for _, v := range rowSum {
                prefix += v
                target := prefix - k
                idx := sort.SearchInts(sorted, target)
                if idx < len(sorted) {
                    cur := prefix - sorted[idx]
                    if cur > ans { ans = cur }
                }
                pos := sort.SearchInts(sorted, prefix)
                sorted = append(sorted, 0)
                copy(sorted[pos+1:], sorted[pos:])
                sorted[pos] = prefix
            }
        }
    }
    return ans
}

### Rust

In [ ]:
use std::collections::BTreeSet;

impl Solution {
    pub fn max_sum_submatrix(matrix: Vec<Vec<i32>>, k: i32) -> i32 {
        let (m, n) = (matrix.len(), matrix[0].len());
        let mut ans = i32::MIN;
        for l in 0..n {
            let mut row_sum = vec![0i64; m];
            for r in l..n {
                for i in 0..m {
                    row_sum[i] += matrix[i][r] as i64;
                }
                let mut sorted = BTreeSet::new();
                sorted.insert(0i64);
                let mut prefix: i64 = 0;
                for &v in &row_sum {
                    prefix += v;
                    let target = prefix - k as i64;
                    if let Some(&val) = sorted.range(target..).next() {
                        ans = ans.max((prefix - val) as i32);
                    }
                    sorted.insert(prefix);
                }
            }
        }
        ans
    }
}

## Example Scenarios

### Scenario 1: Small matrix with exact match
**Input:** `matrix = [[1,0,1],[0,-2,3]], k = 2`  
The sub-rectangle `[[0,1],[-2,3]]` sums to 2 which equals k. **Output:** `2`

### Scenario 2: Single element matrix
**Input:** `matrix = [[2]], k = 3`  
Only one rectangle with sum 2, which is <= 3. **Output:** `2`

### Scenario 3: All negative values
**Input:** `matrix = [[-1,-2],[-3,-4]], k = -1`  
The single element -1 gives the maximum sum <= -1. **Output:** `-1`

### Scenario 4: Full matrix is the answer
**Input:** `matrix = [[1,1],[1,1]], k = 10`  
The entire matrix sums to 4 which is <= 10. **Output:** `4`

### Scenario 5: Need sorted set to beat brute force
**Input:** `matrix = [[5,-4,-3,4],[-3,-4,4,5],[5,1,5,-4]], k = 8`  
We need to find the rectangle with maximum sum not exceeding 8. **Output:** `8`

*Infographic will be added in a future update.*